# asyncstdlib

## Example 1: async builtins over an async generator

In [ ]:
# Example 1: zip / map / enumerate / sum / sorted, each accepting an async iterable
import asyncstdlib as a

async def count(n):
    for i in range(n):
        yield i

pairs = await a.list(a.zip(count(3), "abc"))
assert pairs == [(0, "a"), (1, "b"), (2, "c")]
squares = await a.list(a.map(lambda x: x * x, count(5)))
assert squares == [0, 1, 4, 9, 16] and await a.sum(count(5)) == 10
assert await a.sorted(count(4), reverse=True) == [3, 2, 1, 0]
assert await a.list(a.enumerate(count(2), start=10)) == [(10, 0), (11, 1)]
print("Example 1:", pairs, squares)

## Example 2: async itertools

In [ ]:
# Example 2: islice / takewhile / groupby / batched / chain behave like itertools
import asyncstdlib as a

async def count():
    i = 0
    while True:
        yield i
        i += 1

assert await a.list(a.islice(count(), 2, 8, 2)) == [2, 4, 6]
assert await a.list(a.takewhile(lambda x: x < 3, count())) == [0, 1, 2]
groups = [(k, await a.list(g)) async for k, g in a.groupby(a.islice(count(), 6), key=lambda x: x // 2)]
assert groups == [(0, [0, 1]), (1, [2, 3]), (2, [4, 5])], groups
assert await a.list(a.batched(a.islice(count(), 5), 2)) == [(0, 1), (2, 3), (4,)]
assert await a.list(a.chain([1], a.islice(count(), 2))) == [1, 0, 1]
print("Example 2:", groups)

## Example 3: lru_cache on a coroutine function

In [ ]:
# Example 3: lru_cache caches awaited results, counts hits and misses, and can be cleared
import asyncio
import asyncstdlib as a

calls = []

@a.lru_cache(maxsize=2)
async def slow_square(x):
    calls.append(x)
    await asyncio.sleep(0)
    return x * x

assert [await slow_square(x) for x in (2, 2, 3, 2)] == [4, 4, 9, 4]
info = slow_square.cache_info()
assert calls == [2, 3] and (info.hits, info.misses) == (2, 2), (calls, info)
slow_square.cache_clear()
assert await slow_square(2) == 4 and calls == [2, 3, 2]
print("Example 3:", info)

## Example 4: async context managers

In [ ]:
# Example 4: contextmanager turns an async generator into an async context manager; ExitStack unwinds in reverse
import asyncstdlib as a

events = []

@a.contextmanager
async def resource(name):
    events.append(f"open {name}")
    yield name.upper()
    events.append(f"close {name}")

async with a.ExitStack() as stack:
    got = [await stack.enter_context(resource(n)) for n in ("x", "y")]
assert got == ["X", "Y"]
assert events == ["open x", "open y", "close y", "close x"], events
print("Example 4:", events)

## Example 5: scoped_iter closes the generator early

In [ ]:
# Example 5: asynctools.scoped_iter guarantees the async generator's cleanup runs when the scope exits early
import asyncstdlib as a

closed = []

async def numbers():
    try:
        for i in range(100):
            yield i
    finally:
        closed.append(True)

async with a.scoped_iter(numbers()) as it:
    first = [x async for x in a.islice(it, 3)]
assert first == [0, 1, 2] and closed == [True], (first, closed)
assert await a.reduce(lambda x, y: x * y, a.islice(numbers(), 1, 5)) == 24
print("Example 5:", first, "generator closed:", closed)